# APT Lab 01 — GRPO Ratio Clipping and Gradient Diagnostics

**Learning path:** toy GRPO clipping → diagnostic metric → tested observation → reuse in TRL GRPO.

**Status:** interactive student lab; not a full GRPO trainer. Run cells in order in Google Colab (**Runtime → Run all** only after completing TODOs).

**Goals:** implement the clipped surrogate, compute raw/clipped gradients, explain the sign flip in cosine alignment, and write a diagnostic check. This notebook uses only PyTorch and CPU.

## 0. Setup and reproducibility

Colab normally has PyTorch installed. This cell verifies it rather than silently upgrading the runtime.

In [ ]:
import torch
import torch.nn.functional as F
print('PyTorch:', torch.__version__)
print('Device: CPU (sufficient for this lab)')
torch.manual_seed(7)

## 1. Predict before coding

For the two samples below, the initial probability ratios are `1.3` and `1.0`. Both have advantage `+1`. With `epsilon = 0.2`, which sample should be suppressed by GRPO ratio clipping? Write your prediction here before running the next cell.

> **My prediction:** ...

Recall that clipping the **importance ratio in the loss** is not the same as clipping the **gradient norm**.

## 2. Build a differentiable toy ratio model

We use `log_ratio = log(initial_ratio) + feature · theta`. The feature vectors are an explicit local model of how log probabilities change with parameters. They are **not** a complete normalized token policy, so this is a controlled mechanism demonstration.

In [ ]:
theta = torch.nn.Parameter(torch.zeros(2))
features = torch.tensor([[2.0, 0.0], [0.0, 1.0]])
initial_ratios = torch.tensor([1.3, 1.0])
advantages = torch.tensor([1.0, 1.0])
epsilon = 0.2

log_ratios = torch.log(initial_ratios) + features @ theta
ratios = torch.exp(log_ratios)
print('Ratios:', ratios.detach().tolist())

## 3. Your implementation — two lines

The positive-advantage clipped objective is `min(ratio * advantage, clamp(ratio, 1-epsilon, 1+epsilon) * advantage)`.

Replace the two `...` values. **Don't reveal the reference solution until after attempting this.**

In [ ]:
# TODO 1: multiply each probability ratio by its advantage.
unclipped = ...
# TODO 2: clamp each ratio to [1-epsilon, 1+epsilon], then multiply by its advantage.
clipped = ...

assert isinstance(unclipped, torch.Tensor), 'Implement unclipped first'
assert isinstance(clipped, torch.Tensor), 'Implement clipped first'
assert unclipped.shape == (2,) and clipped.shape == (2,)
print('Unclipped:', unclipped.detach().tolist())
print('Clipped:', clipped.detach().tolist())

## 4. Measure gradients and calculate cosine similarity

For each objective, `autograd.grad` returns the gradient with respect to `theta`. We compare the raw and clipped atomic gradients against a **fixed illustrative composition gradient** `[1, -1]`. A positive cosine means local alignment; negative means local conflict.

In [ ]:
raw_objective = unclipped.sum()
clipped_objective = torch.minimum(unclipped, clipped).sum()
g_raw = torch.autograd.grad(raw_objective, theta, retain_graph=True)[0]
g_clipped = torch.autograd.grad(clipped_objective, theta)[0]
g_composition = torch.tensor([1.0, -1.0])
raw_cos = F.cosine_similarity(g_raw[None], g_composition[None]).item()
clipped_cos = F.cosine_similarity(g_clipped[None], g_composition[None]).item()
print('Raw gradient:', g_raw.tolist())
print('Clipped gradient:', g_clipped.tolist())
print('Raw cosine:', round(raw_cos, 3))
print('Clipped cosine:', round(clipped_cos, 3))

## 5. Diagnostic tests — student exercise

Add an assertion that ensures (a) raw cosine is positive, (b) clipped cosine is negative. Then test the gradient vectors with `torch.testing.assert_close`. The reference expectations for this toy setup are `g_raw=[2.6,1]` and `g_clipped=[0,1]`.

In [ ]:
# TODO 3: replace `pass` with your assertions.
# Helpful examples: assert raw_cos > 0 ; torch.testing.assert_close(g_raw, torch.tensor([...]))
pass

## 6. Turn observations into a simple APT report

This reports a verified **toy clipping sign-flip mechanism**, not a claim that clipping caused a production held-out regression.

In [ ]:
report = {
    'experiment_id': 'toy_grpo_clipping_01',
    'evidence_scope': 'controlled_toy_surrogate',
    'ratios': [float(v) for v in ratios.detach()],
    'clip_epsilon': epsilon,
    'raw_gradient_cosine': raw_cos,
    'clipped_gradient_cosine': clipped_cos,
    'sign_flip': bool(raw_cos > 0 and clipped_cos < 0),
    'diagnosis': 'clipping_changes_gradient_direction_in_this_toy_example',
}
import json
from pathlib import Path
Path('apt_lab01_report.json').write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))

## 7. Investigate — don't just copy outputs

1. Change the first `initial_ratios` value from `1.3` to `1.1`. What happens to the clipped gradient?
2. Change its advantage to `-1` and reason about which side of the clipping interval suppresses negative-advantage gradients.
3. Explain why a sign flip **does not prove** that clipping caused a real held-out accuracy regression.

**Submit to your tutor:** your two TODO lines, the assertion test, outputs, and brief explanation. The next notebook will preserve the report shape while moving to a real TRL `SFTTrainer` experiment with label-mask diagnostics.

## Reference solution (open only after attempting)

For TODO 1–2:

```python
unclipped = ratios * advantages
clipped = torch.clamp(ratios, 1 - epsilon, 1 + epsilon) * advantages
```

For TODO 3:

```python
assert raw_cos > 0
assert clipped_cos < 0
torch.testing.assert_close(g_raw, torch.tensor([2.6, 1.0]), atol=1e-5, rtol=1e-5)
torch.testing.assert_close(g_clipped, torch.tensor([0.0, 1.0]))
```

Run outputs will be generated only when you execute the notebook.